# Debug: Sky Background & Dark Current Discrepancy

Analyse détaillée pour comprendre pourquoi les évolutions avec sky et dark current ne matchent pas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'notebooks'))

from Observation import Observation, load_instruments
from astropy.stats import signal_to_noise_oir_ccd

plt.style.use('seaborn-v0_8-darkgrid')
print("✓ Imports OK")

In [ ]:
# Load instruments
instruments, database = load_instruments()

# Use GALEX FUV (no read noise = simpler)
instrument_name = "GALEX FUV"

# Extract parameters
params = {}
for i, charact in enumerate(instruments['Charact.']):
    if charact and not isinstance(charact, np.ma.core.MaskedConstant):
        value = instruments[instrument_name][i]
        if not isinstance(value, np.ma.core.MaskedConstant):
            params[charact] = value

print(f"Instrument: {instrument_name}")
print(f"\nKey parameters:")
print(f"  Signal: {params['Signal']:.2e} erg/cm²/s/arcsec²/Å")
print(f"  Sky: {params['Sky']:.2e} erg/cm²/s/arcsec²/Å")
print(f"  Dark current: {params['Dark_current']:.4f} e⁻/pix/hour")
print(f"  Read noise: {params['RN']:.2f} e⁻")
print(f"  Collecting area: {params['Collecting_area']:.3f} m²")
print(f"  Throughput: {params['Throughput']:.3f}")
print(f"  QE: {params['QE']:.3f}")
print(f"  Atmosphere: {params['Atmosphere']:.3f}")
print(f"  Pixel scale: {params['pixel_scale']:.2f} arcsec/pix")
print(f"  Dispersion: {params['dispersion']:.2f} Å/pix")
print(f"  Wavelength: {params['wavelength']:.1f} nm")

## 1. Analyse détaillée d'un cas simple

Comparons les valeurs intermédiaires pour un cas de référence.

In [ ]:
# Reference case
t_exp = 1000.0  # seconds

# Generic ETC
obs = Observation(
    instruments=instruments,
    instrument=instrument_name,
    exposure_time=t_exp,
    SNR_res="per pix",
    IFS=False,
    test=False
)

print("="*60)
print("GENERIC ETC - Internal values")
print("="*60)
print(f"SNR: {obs.SNR[obs.i]:.6f}")
print(f"\nSignal:")
print(f"  Signal_el (e⁻): {obs.Signal_el[obs.i]:.6f}")
print(f"\nNoise components:")
print(f"  Sky noise (e⁻): {obs.sky[obs.i]:.6f}")
print(f"  Dark current (e⁻): {obs.Dark_current_f[obs.i]:.6f}")
print(f"  Total noise (e⁻): {obs.Total_noise_final[obs.i]:.6f}")
print(f"\nPixels:")
print(f"  number_pixels_used: {obs.number_pixels_used}")

# Try to access more internal variables
try:
    print(f"\nOther attributes:")
    for attr in ['sky_el', 'Dark_el', 'RN_final', 'flux_fraction']:
        if hasattr(obs, attr):
            val = getattr(obs, attr)
            if hasattr(val, '__getitem__'):
                try:
                    print(f"  {attr}: {val[obs.i]}")
                except:
                    print(f"  {attr}: {val}")
            else:
                print(f"  {attr}: {val}")
except Exception as e:
    print(f"  Error: {e}")

In [ ]:
# Astropy calculation - step by step
print("="*60)
print("ASTROPY SNR - Step by step calculation")
print("="*60)

# Step 1: Convert flux to photons
wavelength_nm = params['wavelength']
E_photon = 1.986e-8 / wavelength_nm  # erg
print(f"\n1. Photon energy at {wavelength_nm} nm: {E_photon:.4e} erg")

signal_flux = params['Signal']  # erg/cm²/s/arcsec²/Å
sky_flux = params['Sky']        # erg/cm²/s/arcsec²/Å

signal_photons_rate = signal_flux / E_photon  # photons/cm²/s/arcsec²/Å
sky_photons_rate = sky_flux / E_photon        # photons/cm²/s/arcsec²/Å

print(f"\n2. Flux to photons:")
print(f"   Signal: {signal_flux:.2e} erg → {signal_photons_rate:.4e} photons/cm²/s/arcsec²/Å")
print(f"   Sky: {sky_flux:.2e} erg → {sky_photons_rate:.4e} photons/cm²/s/arcsec²/Å")

# Step 2: Apply telescope/instrument
collecting_area_cm2 = params['Collecting_area'] * 1e4  # m² → cm²
pixel_area_arcsec2 = params['pixel_scale']**2          # arcsec²
wavelength_range = params['dispersion']                 # Å/pixel
total_throughput = params['Throughput'] * params['QE'] * params['Atmosphere']

print(f"\n3. Telescope/instrument:")
print(f"   Collecting area: {collecting_area_cm2:.1f} cm²")
print(f"   Pixel area: {pixel_area_arcsec2:.2f} arcsec²")
print(f"   Wavelength range: {wavelength_range:.2f} Å/pix")
print(f"   Total throughput: {total_throughput:.4f}")

# Step 3: Calculate electron rates
source_eps = (signal_photons_rate * 
              collecting_area_cm2 * 
              pixel_area_arcsec2 * 
              wavelength_range * 
              total_throughput)

sky_eps = (sky_photons_rate * 
           collecting_area_cm2 * 
           pixel_area_arcsec2 * 
           wavelength_range * 
           total_throughput)

dark_eps = params['Dark_current'] / 3600.0  # e⁻/pix/hour → e⁻/s/pix

print(f"\n4. Electron rates:")
print(f"   source_eps: {source_eps:.6e} e⁻/s")
print(f"   sky_eps: {sky_eps:.6e} e⁻/s/pix")
print(f"   dark_eps: {dark_eps:.6e} e⁻/s/pix")

# Step 4: Total electrons over exposure
n_pixels = 1  # per pixel mode

signal_total = source_eps * t_exp
sky_total = sky_eps * t_exp * n_pixels
dark_total = dark_eps * t_exp * n_pixels
read_total = params['RN']**2 * n_pixels

print(f"\n5. Total electrons (t_exp = {t_exp}s):")
print(f"   Signal: {signal_total:.6f} e⁻")
print(f"   Sky: {sky_total:.6f} e⁻")
print(f"   Dark: {dark_total:.6f} e⁻")
print(f"   Read²: {read_total:.6f} e⁻²")

# Step 5: SNR calculation
noise_variance = signal_total + sky_total + dark_total + read_total
noise_total = np.sqrt(noise_variance)
snr_manual = signal_total / noise_total

print(f"\n6. SNR calculation:")
print(f"   Noise variance: {noise_variance:.6f}")
print(f"   Noise (sqrt): {noise_total:.6f} e⁻")
print(f"   SNR (manual): {snr_manual:.6f}")

# Compare with Astropy function
snr_astropy = signal_to_noise_oir_ccd(
    t=t_exp,
    source_eps=source_eps,
    sky_eps=sky_eps,
    dark_eps=dark_eps,
    rd=params['RN'],
    npix=n_pixels,
    gain=1.0
)
print(f"   SNR (astropy): {snr_astropy:.6f}")

## 2. Comparaison Signal vs Sky

Regardons les ratios signal/sky pour comprendre la discrepancy.

In [ ]:
print("="*60)
print("COMPARISON: Generic ETC vs Astropy")
print("="*60)

print(f"\nGeneric ETC:")
print(f"  Signal: {obs.Signal_el[obs.i]:.6f} e⁻")
print(f"  Sky: {obs.sky[obs.i]:.6f} e⁻")
print(f"  Dark: {obs.Dark_current_f[obs.i]:.6f} e⁻")
print(f"  SNR: {obs.SNR[obs.i]:.6f}")

print(f"\nAstropy:")
print(f"  Signal: {signal_total:.6f} e⁻")
print(f"  Sky: {sky_total:.6f} e⁻")
print(f"  Dark: {dark_total:.6f} e⁻")
print(f"  SNR: {snr_astropy:.6f}")

print(f"\nRatios (Generic/Astropy):")
print(f"  Signal: {obs.Signal_el[obs.i]/signal_total:.4f}")
print(f"  Sky: {obs.sky[obs.i]/sky_total:.4f}")
print(f"  Dark: {obs.Dark_current_f[obs.i]/dark_total:.4f}")
print(f"  SNR: {obs.SNR[obs.i]/snr_astropy:.4f}")

print(f"\n⚠️ Key observation:")
if abs(obs.Signal_el[obs.i]/signal_total - 1) > 0.1:
    print(f"  Signal differs significantly!")
if abs(obs.sky[obs.i]/sky_total - 1) > 0.1:
    print(f"  Sky differs significantly!")
if abs(obs.Dark_current_f[obs.i]/dark_total - 1) > 0.1:
    print(f"  Dark differs significantly!")

## 3. Test avec variation du Sky (plus de points)

Exporter les données pour analyse.

In [ ]:
# Test sky variation with more points
sky_factors = np.logspace(-2, 2, 30)  # 0.01x to 100x nominal

sky_flux_nominal = params['Sky']
signal_flux = params['Signal']
exposure_time = params.get('exposure_time', 1000.0)

data = []

for factor in sky_factors:
    sky_flux = sky_flux_nominal * factor
    
    # Modify instrument
    original_sky = instruments[instrument_name][list(instruments['Charact.']).index('Sky')]
    instruments[instrument_name][list(instruments['Charact.']).index('Sky')] = sky_flux
    
    # Generic ETC
    obs = Observation(
        instruments=instruments,
        instrument=instrument_name,
        exposure_time=exposure_time,
        SNR_res="per pix",
        IFS=False,
        test=False
    )
    snr_generic = obs.SNR[obs.i]
    signal_generic = obs.Signal_el[obs.i]
    sky_generic = obs.sky[obs.i]
    dark_generic = obs.Dark_current_f[obs.i]
    noise_generic = obs.Total_noise_final[obs.i]
    
    # Restore
    instruments[instrument_name][list(instruments['Charact.']).index('Sky')] = original_sky
    
    # Astropy
    sky_photons = sky_flux / E_photon
    sky_eps = (sky_photons * collecting_area_cm2 * pixel_area_arcsec2 * 
               wavelength_range * total_throughput)
    
    snr_astropy = signal_to_noise_oir_ccd(
        t=exposure_time,
        source_eps=source_eps,
        sky_eps=sky_eps,
        dark_eps=dark_eps,
        rd=params['RN'],
        npix=1,
        gain=1.0
    )
    
    sky_astropy = sky_eps * exposure_time
    
    data.append({
        'sky_factor': factor,
        'sky_flux': sky_flux,
        'snr_generic': snr_generic,
        'snr_astropy': snr_astropy,
        'rel_diff_pct': 100 * (snr_generic - snr_astropy) / snr_astropy,
        'signal_generic': signal_generic,
        'sky_generic': sky_generic,
        'sky_astropy': sky_astropy,
        'sky_ratio': sky_generic / sky_astropy if sky_astropy > 0 else np.nan,
        'dark_generic': dark_generic,
        'noise_generic': noise_generic
    })

df_sky = pd.DataFrame(data)
print(f"Sky variation test: {len(df_sky)} points")
df_sky.head(10)

In [ ]:
# Plot sky comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SNR comparison
ax = axes[0, 0]
ax.loglog(df_sky['sky_factor'], df_sky['snr_generic'], 'o-', label='Generic ETC')
ax.loglog(df_sky['sky_factor'], df_sky['snr_astropy'], 's--', label='Astropy', alpha=0.7)
ax.set_xlabel('Sky Factor')
ax.set_ylabel('SNR')
ax.set_title('SNR vs Sky Factor')
ax.legend()
ax.grid(True, alpha=0.3)

# Relative difference
ax = axes[0, 1]
ax.semilogx(df_sky['sky_factor'], df_sky['rel_diff_pct'], 'o-', color='red')
ax.axhline(0, color='black', linestyle='--')
ax.axhline(15, color='orange', linestyle=':')
ax.axhline(-15, color='orange', linestyle=':')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Relative Difference (%)')
ax.set_title('Relative Difference vs Sky Factor')
ax.grid(True, alpha=0.3)

# Sky electrons comparison
ax = axes[1, 0]
ax.loglog(df_sky['sky_factor'], df_sky['sky_generic'], 'o-', label='Generic ETC sky (e⁻)')
ax.loglog(df_sky['sky_factor'], df_sky['sky_astropy'], 's--', label='Astropy sky (e⁻)', alpha=0.7)
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Sky electrons')
ax.set_title('Sky Electrons Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Sky ratio
ax = axes[1, 1]
ax.semilogx(df_sky['sky_factor'], df_sky['sky_ratio'], 'o-', color='green')
ax.axhline(1, color='black', linestyle='--')
ax.set_xlabel('Sky Factor')
ax.set_ylabel('Sky Ratio (Generic/Astropy)')
ax.set_title('Sky Electrons Ratio')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSky ratio (Generic/Astropy): {df_sky['sky_ratio'].mean():.4f} ± {df_sky['sky_ratio'].std():.4f}")

## 4. Test avec variation du Dark Current

In [ ]:
# Test dark current variation
dark_factors = np.logspace(-2, 2, 30)  # 0.01x to 100x nominal

dark_current_nominal = params['Dark_current']

data_dark = []

for factor in dark_factors:
    dark_current = dark_current_nominal * factor
    
    # Modify instrument
    original_dc = instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')]
    instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')] = dark_current
    
    # Generic ETC
    obs = Observation(
        instruments=instruments,
        instrument=instrument_name,
        exposure_time=exposure_time,
        SNR_res="per pix",
        IFS=False,
        test=False
    )
    snr_generic = obs.SNR[obs.i]
    dark_generic = obs.Dark_current_f[obs.i]
    
    # Restore
    instruments[instrument_name][list(instruments['Charact.']).index('Dark_current')] = original_dc
    
    # Astropy
    dark_eps_var = dark_current / 3600.0
    
    snr_astropy = signal_to_noise_oir_ccd(
        t=exposure_time,
        source_eps=source_eps,
        sky_eps=sky_eps,
        dark_eps=dark_eps_var,
        rd=params['RN'],
        npix=1,
        gain=1.0
    )
    
    dark_astropy = dark_eps_var * exposure_time
    
    data_dark.append({
        'dark_factor': factor,
        'dark_current': dark_current,
        'snr_generic': snr_generic,
        'snr_astropy': snr_astropy,
        'rel_diff_pct': 100 * (snr_generic - snr_astropy) / snr_astropy,
        'dark_generic': dark_generic,
        'dark_astropy': dark_astropy,
        'dark_ratio': dark_generic / dark_astropy if dark_astropy > 0 else np.nan
    })

df_dark = pd.DataFrame(data_dark)
print(f"Dark current variation test: {len(df_dark)} points")
df_dark.head(10)

In [ ]:
# Plot dark comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SNR comparison
ax = axes[0, 0]
ax.loglog(df_dark['dark_factor'], df_dark['snr_generic'], 'o-', label='Generic ETC')
ax.loglog(df_dark['dark_factor'], df_dark['snr_astropy'], 's--', label='Astropy', alpha=0.7)
ax.set_xlabel('Dark Factor')
ax.set_ylabel('SNR')
ax.set_title('SNR vs Dark Factor')
ax.legend()
ax.grid(True, alpha=0.3)

# Relative difference
ax = axes[0, 1]
ax.semilogx(df_dark['dark_factor'], df_dark['rel_diff_pct'], 'o-', color='red')
ax.axhline(0, color='black', linestyle='--')
ax.axhline(15, color='orange', linestyle=':')
ax.axhline(-15, color='orange', linestyle=':')
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Relative Difference (%)')
ax.set_title('Relative Difference vs Dark Factor')
ax.grid(True, alpha=0.3)

# Dark electrons comparison
ax = axes[1, 0]
ax.loglog(df_dark['dark_factor'], df_dark['dark_generic'], 'o-', label='Generic ETC dark (e⁻)')
ax.loglog(df_dark['dark_factor'], df_dark['dark_astropy'], 's--', label='Astropy dark (e⁻)', alpha=0.7)
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Dark electrons')
ax.set_title('Dark Electrons Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Dark ratio
ax = axes[1, 1]
ax.semilogx(df_dark['dark_factor'], df_dark['dark_ratio'], 'o-', color='purple')
ax.axhline(1, color='black', linestyle='--')
ax.set_xlabel('Dark Factor')
ax.set_ylabel('Dark Ratio (Generic/Astropy)')
ax.set_title('Dark Electrons Ratio')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDark ratio (Generic/Astropy): {df_dark['dark_ratio'].mean():.4f} ± {df_dark['dark_ratio'].std():.4f}")

## 5. Export CSV pour analyse

In [ ]:
# Save to CSV
df_sky.to_csv('./debug_sky_variation.csv', index=False)
df_dark.to_csv('./debug_dark_variation.csv', index=False)

print("✓ Saved debug_sky_variation.csv")
print("✓ Saved debug_dark_variation.csv")

# Summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"\nSky test:")
print(f"  Sky ratio (Generic/Astropy): {df_sky['sky_ratio'].mean():.4f}")
print(f"  Max rel diff: {df_sky['rel_diff_pct'].abs().max():.2f}%")

print(f"\nDark test:")
print(f"  Dark ratio (Generic/Astropy): {df_dark['dark_ratio'].mean():.4f}")
print(f"  Max rel diff: {df_dark['rel_diff_pct'].abs().max():.2f}%")

## 6. Hypothèses sur la discrepancy

Basé sur les ratios observés, voici les hypothèses possibles :

### Si sky_ratio ≠ 1 :
- **Facteur de pixels** : Generic ETC multiplie peut-être par un nombre de pixels différent
- **Unités** : Le sky dans Generic ETC n'est pas en erg/cm²/s/arcsec²/Å
- **Résolution spectrale** : Generic ETC intègre peut-être sur plus de pixels spectraux

### Si dark_ratio ≠ 1 :
- **Conversion heure → seconde** : Vérifier le facteur 3600
- **Nombre de pixels** : Dark current multiplié par elem_size au lieu de 1

### À vérifier dans Observation.py :
1. Comment `self.sky` est calculé
2. Comment `self.Dark_current_f` est calculé
3. Le rôle de `elem_size` et `number_pixels_used`